# RSNA Knee Abnormality Detection - EDA(Exploratory Data Analysis) 【English + 日本語】

### Credits:

- https://www.kaggle.com/code/liamob96/knee-abnormality-eda
- https://www.kaggle.com/code/stevenleehans/rsna-knee-101-read-this-before-you-start

## Objective(目的)

The goal of this competition is to predict 12 knee abnormalities from MRI studies. The training data also contain multilingual radiology reports, which may provide supervision for studies without structured target labels.

このコンペティションの目的は、MRI研究から12の膝の異常を予測することです。トレーニングデータには、多言語の放射線レポートも含まれており、構造化されたターゲットラベルがない研究に対して監督を提供する可能性があります。


### 1. Data Overview(データ概要)

This notebook examines the dataset at the study, series, report and image levels before modelling. Particular attention is given to the unusually small gold-labelled subset, possible leakage routes, MRI acquisition variability and a reproducible validation design.

> **Important:** missing target values mean _unlabelled_, not _negative_. Series and slices from the same study must remain together, and patient-level grouping must be investigated separately.

#### Dataset at a glance

| Item                        | Visible competition data |
| --------------------------- | -----------------------: |
| Training studies            |                    4,407 |
| Fully gold-labelled studies |                       58 |
| Report-only studies         |                    4,349 |
| Prediction targets          |                       12 |
| Training MRI series         |                   24,371 |
| Visible test studies        |                        3 |
| Predicted report languages  |                        9 |

このノートブックでは、モデリングの前に、研究、シリーズ、レポート、および画像レベルでデータセットを調べます。特に、異常に小さいゴールドラベル付きサブセット、可能なリークルート、MRI取得の変動性、および再現可能な検証設計に注意が払われます。

> **重要:** ターゲット値が欠落している場合は、*ラベルなし*を意味し、*負*を意味しません。同じ研究からのシリーズとスライスは一緒に保持する必要があり、患者レベルのグループ化は別途調査する必要があります。


## 2. Read the data(データの読み込み)


### 2.1. Import libraries(ライブラリのインポート)


In [13]:
# ===================================
# 2.1. Imports and settings(インポートと設定)
# ===================================

from pathlib import Path
from collections import Counter
import os
import warnings
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 10

# Change the path to your dataset directory(データセットディレクトリへのパスを変更してください)
# Kaggle
# DATA_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection/")
DATA_ROOT = Path("E:/rsna-knee-abnormality-detection")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

sns.set_theme(style="whitegrid", palette="muted", context="notebook")

print(f"NumPy:      {metadata.version('numpy')}")
print(f"Pandas:     {metadata.version('pandas')}")
print(f"Matplotlib: {metadata.version('matplotlib')}")
print(f"Seaborn:    {metadata.version('seaborn')}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Input root:  {DATA_ROOT}")

NumPy:      2.3.5
Pandas:     2.3.3
Matplotlib: 3.10.0
Seaborn:    0.13.2
Random seed: 10
Input root:  E:\rsna-knee-abnormality-detection


### 2.2. Load the data(データの読み込み)


In [19]:
# ===================================
# 2.2.1. Load data(データの読み込み)
# ===================================

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data root directory '{DATA_ROOT}' does not exist.")


def describe_entry(path: Path) -> dict:
    """Describe a file or directory entry."""
    if path.is_file():
        return {
            "name": path.name,
            "type": "file",
            "size_mb": path.stat().st_size / 1024**2,
            "children": np.nan,
        }

    children = list(path.iterdir())

    return {
        "name": path.name,
        "type": "directory",
        "size_mb": np.nan,
        "children": len(children),
    }


input_summary = pd.DataFrame([describe_entry(path) for path in DATA_ROOT.iterdir()])

display(input_summary)

,name,type,size_mb,children
0,sample_submission.csv,file,0.0004,NaN
1,test.csv,file,0.0002,NaN
2,test_series.csv,file,0.0021,NaN
3,test_series,directory,NaN,3.0000
4,train.csv,file,5.4264,NaN
5,train_series.csv,file,3.2986,NaN
6,train_series,directory,NaN,4407.0000


In [12]:
# ============================================================
# 2.2.2 Load the tabular competition files
# ============================================================

TRAIN_PATH = DATA_ROOT / "train.csv"
TEST_PATH = DATA_ROOT / "test.csv"
TRAIN_SERIES_PATH = DATA_ROOT / "train_series.csv"
TEST_SERIES_PATH = DATA_ROOT / "test_series.csv"
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"

table_paths = {
    "train": TRAIN_PATH,
    "test": TEST_PATH,
    "train_series": TRAIN_SERIES_PATH,
    "test_series": TEST_SERIES_PATH,
    "sample_submission": SAMPLE_SUBMISSION_PATH,
}

missing_files = [str(path) for path in table_paths.values() if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "The following expected files were not found:\n" + "\n".join(missing_files)
    )

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train_series = pd.read_csv(TRAIN_SERIES_PATH)
test_series = pd.read_csv(TEST_SERIES_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

tables = {
    "train": train,
    "test": test,
    "train_series": train_series,
    "test_series": test_series,
    "sample_submission": sample_submission,
}

print("All tabular files loaded successfully.")

All tabular files loaded successfully.


## 3. Tablular data exploration(表形式データの探索)

We will first explore the tabular data, which contains the study-level labels and metadata.

最初は、研究レベルのラベルとメタデータを含む表形式のデータを探索します。

First we need to know how many rows and collumns each table contains.

まずは、各テーブルに何行何列あるかを知る必要があります。


### 3.1. Check table rows and columns count (テーブルの行数と列数の確認)


In [15]:
# ===================================
# 3.1. Check table rows and columns count (テーブルの行数と列数の確認)
# ===================================

table_overview = pd.DataFrame(
    {
        "table_name": list(tables.keys()),
        "num_rows": [df.shape[0] for df in tables.values()],
        "num_columns": [df.shape[1] for df in tables.values()],
    }
)

display(table_overview)

,table_name,num_rows,num_columns
0,train,4407,14
1,test,3,1
2,train_series,24371,5
3,test_series,15,5
4,sample_submission,3,13


### 3.2. Check Columns, data types and data-quality (列とデータ型の確認)


In [18]:
# ===================================
# 3.2. Check Columns, data types and data-quality (列とデータ型の確認)
# ===================================

table_columns_info = {
    table_name: df.dtypes.reset_index()
    .rename(
        columns={
            "index": "column_name",
            0: "data_type",
            1: "num_missing",
            2: "num_unique",
        }
    )
    .assign(
        num_missing=df.isnull().sum().values,
        num_unique=df.nunique().values,
    )
    for table_name, df in tables.items()
}

for table_name, columns_info in table_columns_info.items():
    print(f"Table: {table_name}")
    display(columns_info)
    print()

Table: train


,column_name,data_type,num_missing,num_unique
0,StudyInstanceUID,object,0,4407
1,Report,object,0,4276
2,ACL,float64,4349,2
3,MCL,float64,4349,2
4,Medial Meniscus,float64,4349,2
5,Lateral Meniscus,float64,4349,2
6,Medial OA,float64,4349,2
7,Lateral OA,float64,4349,2
8,PF OA,float64,4349,2
9,Effusion,float64,4349,2



Table: test


,column_name,data_type,num_missing,num_unique
0,StudyInstanceUID,object,0,3



Table: train_series


,column_name,data_type,num_missing,num_unique
0,StudyInstanceUID,object,0,4407
1,SeriesInstanceUID,object,0,24371
2,Fluid_Sensitive,int64,0,2
3,Fat_Suppression,int64,0,2
4,Anatomical_Plane,object,0,3



Table: test_series


,column_name,data_type,num_missing,num_unique
0,StudyInstanceUID,object,0,3
1,SeriesInstanceUID,object,0,15
2,Fluid_Sensitive,int64,0,2
3,Fat_Suppression,int64,0,2
4,Anatomical_Plane,object,0,3



Table: sample_submission


,column_name,data_type,num_missing,num_unique
0,StudyInstanceUID,object,0,3
1,ACL,float64,0,1
2,MCL,float64,0,1
3,Medial Meniscus,float64,0,1
4,Lateral Meniscus,float64,0,1
5,Medial OA,float64,0,1
6,Lateral OA,float64,0,1
7,PF OA,float64,0,1
8,Effusion,float64,0,1
9,Synovitis,float64,0,1


### 3.3. Preview each table (各テーブルのプレビュー)


In [20]:
# ===================================
# 3.3 Preview each table (各テーブルのプレビュー)
# ===================================

for table_name, df in tables.items():
    print(f"Table: {table_name}")
    display(df.head())
    print()

Table: train


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Table: test


,StudyInstanceUID
0,1.2.826.0.1.3680043.8.498.10047035057544427318...
1,1.2.826.0.1.3680043.8.498.10062861783145312629...
2,1.2.826.0.1.3680043.8.498.10067514707072572280...



Table: train_series


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,0,0,Coronal
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,0,0,Sagittal



Table: test_series


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.11580656442259111255...,0,0,Axial
1,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.17811502614030631664...,0,0,Sagittal
2,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.30565395595045942404...,0,0,Sagittal
3,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.32856494541816845805...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.44334485654554877495...,1,1,Axial



Table: sample_submission


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000


### 3.4. Identify Target columns and labeled studies (ターゲット列とラベル付き研究の特定)

Now we will identify the target columns and labeled studies. The target columns are the 12 knee abnormalities that we need to predict. The labeled studies are the studies that have gold labels for these abnormalities.

これで、ターゲット列とラベル付き研究を特定します。ターゲット列は、予測する必要がある12の膝の異常です。ラベル付き研究は、これらの異常に対してゴールドラベルを持つ研究です。


In [22]:
# ===================================
# 3.4. Identify target columns and labeled studies (ターゲット列とラベル付き研究の特定)
# ===================================

ID_COLUMN = "StudyInstanceUID"

TARGET_COLUMNS = [column for column in sample_submission.columns if column != ID_COLUMN]

print("Target columns:")
print(TARGET_COLUMNS)


labels_per_row = train[TARGET_COLUMNS].notna().sum(axis=1)

label_status = pd.Series(
    np.select(
        [
            labels_per_row == 0,
            labels_per_row == len(TARGET_COLUMNS),
        ],
        [
            "Unlabelled",
            "Fully labelled",
        ],
        default="Partially labelled",
    ),
    name="label_status",
)

label_status_summary = (
    label_status.value_counts()
    .rename_axis("label_status")
    .reset_index(name="number_of_studies")
)

label_status_summary["percentage"] = (
    label_status_summary["number_of_studies"] / len(train) * 100
)

display(label_status_summary)

Target columns:
['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


,label_status,number_of_studies,percentage
0,Unlabelled,4349,98.6839
1,Fully labelled,58,1.3161


### Key Findings(主な発見)

Only **58 out of 4,407 studies(1.3%)** have gold labels for the 12 knee abnormalities. The remaining **4,349 studies (98.7%)** are unlabelled and only have radiology reports.

There are no partially labelled studies: each either has all 12 labels or none.
This is intentional feature of this competition, it is not missing data error. There 58 studies form a small **gold-labelled subset**.

Missing Targets must never be converted directly into `NaN` or `0`. They are not negative labels, they are simply unlabelled.
In Further Exploration we will find out what importance the reports holds for unrevealing the labels of the unlabelled studies.

58件の研究のうち、12件の膝の異常に対してゴールドラベルが付いているのは**4,407件中58件（1.3％）**のみです。残りの**4,349件（98.7％）**はラベルが付いておらず、放射線レポートのみがあります。

半分的にラベル付けされた研究はありません。各研究には、12のラベルすべてがあるか、まったくないかのいずれかです。これは、このコンペティションの意図的な特徴であり、欠落データのエラーではありません。58件の研究は、小さな**ゴールドラベル付きサブセット**を形成します。

ラベルが欠落している場合は、直接`NaN`または`0`に変換してはいけません。これは負のラベルではなく、単にラベルが付いていないだけです。さらに探索を進めると、ラベルが付いていない研究のラベルを明らかにするために、レポートがどのような重要性を持っているかがわかります。


### 3.5. Check the distribution of labels(ラベルの分布の確認)

Now we will check the distribution of labels for the 12 knee abnormalities of the 58 gold-labelled studies.

In [23]:
# ===================================
# 3.5. Check the distribution of labels(ラベルの分布の確認)
# ===================================

fully_labelled_studies = labels_per_row == len(TARGET_COLUMNS)
labelled_train = train.loc[fully_labelled_studies].copy()

target_summary_rows = []

for target in TARGET_COLUMNS:
    values = labelled_train[target].dropna()

    target_summary_rows.append(
        {
            "target": target,
            "labelled_studies": len(values),
            "positive_cases": int((values == 1).sum()),
            "negative_cases": int((values == 0).sum()),
            "positive_percentage": (values == 1).mean() * 100,
            "unique_values": sorted(values.unique().tolist()),
        }
    )

target_summary = (
    pd.DataFrame(target_summary_rows)
    .sort_values("positive_percentage", ascending=False)
    .reset_index(drop=True)
)

display(target_summary)

,target,labelled_studies,positive_cases,negative_cases,positive_percentage,unique_values
0,Effusion,58,35,23,60.3448,"[0.0, 1.0]"
1,Synovitis,58,27,31,46.5517,"[0.0, 1.0]"
2,Medial Meniscus,58,26,32,44.8276,"[0.0, 1.0]"
3,ACL,58,24,34,41.3793,"[0.0, 1.0]"
4,Lateral Meniscus,58,23,35,39.6552,"[0.0, 1.0]"
5,PF OA,58,21,37,36.2069,"[0.0, 1.0]"
6,Contusion,58,19,39,32.7586,"[0.0, 1.0]"
7,Fracture,58,18,40,31.0345,"[0.0, 1.0]"
8,Medial OA,58,15,43,25.8621,"[0.0, 1.0]"
9,Baker's,58,12,46,20.6897,"[0.0, 1.0]"


### 3.6. Validate target values(ターゲット値の検証)

In [25]:
# ===================================
# 3.6. Validate target values(ターゲット値の検証)
# ===================================

observed_target_values = set(labelled_train[TARGET_COLUMNS].to_numpy().ravel())

expected_target_values = {0.0, 1.0}
unexpected_target_values = observed_target_values - expected_target_values

assert (
    not unexpected_target_values
), f"Unexpected target values found: {unexpected_target_values}"

print("Target validation passed.")
print("All labelled target values are binary: 0 or 1.")

Target validation passed.
All labelled target values are binary: 0 or 1.


### Key Findings(主な発見)

All 12 target columns are binary, with values of either `0` or `1`. There are no missing values in the gold-labelled studies.

The distribution of positive and negative cases varies across the different target columns.

Across all targets, there are 240 positive labels among 58 studies, equivalent to approximately **4.14 positive abnormalities per study**. The targets are therefore not mutually exclusive: an individual study can contain several abnormalities.

Because the labelled sample is very small, these percentages are uncertain and should not be interpreted as estimates of prevalence in the full dataset or the general population.
全ての12のターゲット列は二値であり、値は`0`または`1`のいずれかです。ゴールドラベル付き研究には欠損値はありません。

正例と負例の分布は、異なるターゲット列によって異なります。

すべてのターゲットにおいて、58件の研究の中で240件の陽性ラベルがあり、これは1件の研究あたり約**4.14件の陽性異常**に相当します。したがって、ターゲットは相互排他的ではありません。個々の研究には複数の異常が含まれる可能性があります。

ラベル付きサンプルが非常に小さいため、これらの割合は不確かであり、完全なデータセットや一般集団における有病率の推定値として解釈すべきではありません。
  
  


## 4. Understanding the competition file structure(コンペティションのファイル構造の理解)

We first inspect the directory structure and file types before loading any data.

This is important because medical-imaging competitions often contain several levels of organisation, such as:

- patients
- studies
- series
- image files
- tabular metadata or labels

私たちは、データをロードする前に、ディレクトリ構造とファイルタイプを最初に検査します。

これは、医療画像コンペティションには、次のような複数の組織レベルが含まれることが多いため、重要です。


In [10]:
# ===================================
# 2. Load data(データの読み込み)
# ===================================

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data root directory '{DATA_ROOT}' does not exist.")


def describe_entry(path: Path) -> dict:
    """Describe a file or directory entry."""
    if path.is_file():
        return {
            "name": path.name,
            "type": "file",
            "size_mb": path.stat().st_size / 1024**2,
            "children": np.nan,
        }

    children = list(path.iterdir())

    return {
        "name": path.name,
        "type": "directory",
        "size_mb": np.nan,
        "children": len(children),
    }


input_summary = pd.DataFrame([describe_entry(path) for path in DATA_ROOT.iterdir()])

display(input_summary)

,name,type,size_mb,children
0,sample_submission.csv,file,0.0004,NaN
1,test.csv,file,0.0002,NaN
2,test_series.csv,file,0.0021,NaN
3,test_series,directory,NaN,3.0000
4,train.csv,file,5.4264,NaN
5,train_series.csv,file,3.2986,NaN
6,train_series,directory,NaN,4407.0000
